<a href="https://colab.research.google.com/github/ishaanprasad1/dorm-room-deals/blob/main/dorm_room_deals_REC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/ishaanprasad1/dorm-room-deals/blob/main/dorm_room_deals_REC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### *[MIST 4600] Fall 2026 · GitHub Project · Ishaan Prasad*
# **Dorm Room Deals**

**My app helps a college student on a tight budget decide which Steam game to buy this week by finding well-reviewed games that fit their genre, price limit, and laptop, then showing which are on sale and which are hidden gems.**

| # | Sub-question | Tools |
|---|---|---|
| 1 | **Fit:** which highly rated games (85%+ positive reviews) match the student's genre and price limit and run on their laptop (Windows, Mac, or Linux)? | pandas |
| 2 | **Deals:** which of those games are on sale today, and how close is today's price to their lowest price ever? | CheapShark API, looked up by `appid` |
| 3 | **Hidden gems:** which of them are lesser known (under 1 million owners) but rated as highly by players as the big hits? | pandas |

**Requirements checklist:** a user and a decision (above) · 3 sub-questions · a function for each sub-question (`find_fits`, `find_deals`, `compare_hidden_gems`) · a class the app uses (`DealPick`, Cell 4) · an outside tool (CheapShark API) · runs from top to bottom · a recommendation at the end.

---
## Before you run it

1. Put this notebook and `steam_games.csv` in your Google Drive (the setup cell finds the file in My Drive or in a folder up to 3 levels down).
2. No API key is needed: CheapShark is free and works without one.
3. Runtime → **Run all**. Allow access to Google Drive when Colab asks. Cell 5 asks CheapShark about 5 games, so it takes about 15 seconds.

## Setup: find and load the data

Connects to Google Drive, finds `steam_games.csv`, and reads the 1,000 Steam games into `games`. It also imports `requests` (to call CheapShark) and `time` (to pause between calls).

In [32]:
from google.colab import drive
drive.mount("/content/drive")

import glob
import os
import time
import pandas as pd
import requests

# Look for steam_games.csv in My Drive and in folders up to 3 levels down
found = []
for pattern in ["", "*/", "*/*/", "*/*/*/"]:
    found = found + glob.glob("/content/drive/MyDrive/" + pattern + "steam_games.csv")

if len(found) == 0:
    print("❌ Can't find steam_games.csv in your Drive. Upload it next to this notebook and run this cell again.")
else:
    FOLDER = os.path.dirname(found[0]) + "/"
    games = pd.read_csv(FOLDER + "steam_games.csv")
    print(f"✅ Loaded {len(games)} games from:", FOLDER)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Loaded 1000 games from: /content/drive/MyDrive/


## Cell 1 · The student's choices

The student fills in three choices here. Everything below uses them, so changing one line changes the whole answer.

* `GENRE` must be spelled the way Steam spells it, with a capital letter: for example `"Indie"`, `"Action"`, `"RPG"`, or `"Strategy"`.
* `MAX_PRICE` is the most the student wants to spend, in dollars.
* `SYSTEM` is the student's laptop: `"windows"`, `"mac"`, or `"linux"`.

In [33]:
GENRE = "Indie"
MAX_PRICE = 20
SYSTEM = "mac"

## Cell 2 · Sub-question 1: Fit

`find_fits` works like a funnel. It starts with all 1,000 games and keeps only the ones that:

1. have at least 85% positive reviews,
2. list the student's genre (`na=False` skips the few games with no genre),
3. cost no more than the student's price limit, and
4. run on the student's laptop.

It returns the whole table of games that fit, because Sub-questions 2 and 3 need its other columns (like `appid` and `owners_low`).

In [34]:
def find_fits(games, genre, max_price, system):
    fits = games[games["pct_pos_total"] >= 85]
    fits = fits[fits["genres"].str.contains(genre, na=False)]
    fits = fits[fits["price"] <= max_price]
    fits = fits[fits[system] == True]
    return fits

fits = find_fits(games, GENRE, MAX_PRICE, SYSTEM)
print(len(fits), "games fit the student's choices")
print(fits[["name", "price", "pct_pos_total"]].head(10))

154 games fit the student's choices
                      name  price  pct_pos_total
6                 Terraria   9.99             97
8              Garry's Mod   5.99             96
13          Stardew Valley  14.99             98
17  Euro Truck Simulator 2  19.99             97
25                Unturned   0.00             91
31                 Valheim  19.99             94
35           Geometry Dash   4.99             93
36           Hollow Knight  14.99             97
39   Don't Starve Together  14.99             95
51         Project Zomboid  19.99             94


## Cell 3 · Asking CheapShark about one game

`check_price` uses the same two CheapShark requests as Module 5, Exercises 9 to 11:

1. `time.sleep(1)` waits 1 second, so we don't send requests too fast.
2. **Request 1** asks with `{"steamAppID": appid}`. CheapShark answers with a list with one dictionary in it. If the request failed, or the list is empty (CheapShark doesn't sell that game), the function returns `None`.
3. From `data[0]` we take `"cheapest"`, the cheapest price in any store today, and turn the text into a number with `float()`.
4. **Request 2** asks with `{"id": ...}`, the CheapShark ID (`"gameID"`), and gives the lowest price ever, under `"cheapestPriceEver"`, then `"price"`.
5. It returns both prices in a list: `[today, lowest]`.

Every request sends `headers=HEADERS`, because CheapShark blocks requests that don't say what program sent them.

In [35]:
URL = "https://www.cheapshark.com/api/1.0/games"
HEADERS = {"User-Agent": "MIST4600-dorm-room-deals"}

def check_price(appid):
    time.sleep(1)
    response = requests.get(URL, params={"steamAppID": appid}, headers=HEADERS)
    if response.status_code != 200:
        return None
    data = response.json()
    if len(data) == 0:
        return None
    today = float(data[0]["cheapest"])

    response2 = requests.get(URL, params={"id": data[0]["gameID"]}, headers=HEADERS)
    data2 = response2.json()
    lowest = float(data2["cheapestPriceEver"]["price"])
    return [today, lowest]

Try it on one game, Stardew Valley (`appid` 413150). You should see two prices: today's and the lowest ever.

In [36]:
print(check_price(413150))

[14.99, 5.99]


## Cell 4 · A class for each pick

Each game checked with CheapShark becomes a `DealPick` object with its name, its regular price (from `steam_games.csv`), its price today, and its lowest price ever.

* `on_sale()` returns `True` if today's price is lower than the regular price.
* `card()` returns one line of text for the student. `:.2f` shows a price with 2 decimals, like `4.99`.

In [37]:
class DealPick:
    def __init__(self, name, regular_price, today_price, lowest_price):
        self.name = name
        self.regular_price = regular_price
        self.today_price = today_price
        self.lowest_price = lowest_price

    def on_sale(self):
        return self.today_price < self.regular_price

    def card(self):
        return f"{self.name}: ${self.today_price:.2f} today, regular ${self.regular_price:.2f}, lowest ever ${self.lowest_price:.2f}"

Try the class with a made-up pick:

In [38]:
test = DealPick("Test Game", 14.99, 7.49, 5.99)
print(test.card())
print(test.on_sale())

Test Game: $7.49 today, regular $14.99, lowest ever $5.99
True


## Cell 5 · Sub-question 2: Deals

`find_deals` checks the best games that fit:

1. It keeps the games that fit and cost money (free games are never on sale) and takes the 5 with the best reviews. Only 5 games are sent to CheapShark, not all of them.
2. For each game, it calls `check_price`. If CheapShark has no answer (`None`), it uses the regular price for both prices, so the game still shows up.
3. Each game becomes a `DealPick`, and the function returns the list of picks.

Then we print each pick's card (today's price next to the regular price and the lowest price ever), and the picks that are on sale today.

In [39]:
def find_deals(fits):
    paid = fits[fits["price"] > 0]
    shortlist = paid.sort_values("pct_pos_total", ascending=False).head(5)
    picks = []
    # AI helped: to_dict("records") turns the table into a list of
    # dictionaries, one per row (like the Week 7 list of dictionaries)
    for game in shortlist.to_dict("records"):
        prices = check_price(game["appid"])
        if prices is None:
            prices = [game["price"], game["price"]]
        picks.append(DealPick(game["name"], game["price"], prices[0], prices[1]))
    return picks

picks = find_deals(fits)

for pick in picks:
    print(pick.card())

print("On sale today:")
for pick in picks:
    if pick.on_sale():
        print("-", pick.name)

Stardew Valley: $14.99 today, regular $14.99, lowest ever $5.99
OneShot: $9.99 today, regular $9.99, lowest ever $0.92
Vampire Survivors: $4.99 today, regular $4.99, lowest ever $2.69
Balatro: $10.04 today, regular $14.99, lowest ever $10.08
Katana ZERO: $8.99 today, regular $14.99, lowest ever $7.49
On sale today:
- Balatro
- Katana ZERO


## Cell 6 · Sub-question 3: Hidden gems

`compare_hidden_gems` splits the games that fit into two groups by their number of owners (`owners_low`):

* **hidden gems:** under 1 million owners
* **big hits:** 1 million owners or more

It first drops games where `owners_low` is 0, because 0 means the owner count is missing, not that nobody owns the game. Then it prints how many games are in each group and their average percent of positive reviews: are the lesser-known games rated as highly as the big hits? It returns the hidden gems, and we print the 5 best-reviewed ones.

In [40]:
def compare_hidden_gems(fits):
    known = fits[fits["owners_low"] > 0]
    gems = known[known["owners_low"] < 1000000]
    hits = known[known["owners_low"] >= 1000000]
    print("Hidden gems:", len(gems), "games, average rating", round(gems["pct_pos_total"].mean(), 1))
    print("Big hits:", len(hits), "games, average rating", round(hits["pct_pos_total"].mean(), 1))
    return gems

gems = compare_hidden_gems(fits)
best_gems = gems.sort_values("pct_pos_total", ascending=False).head(5)
print(best_gems[["name", "price", "pct_pos_total"]])

Hidden gems: 27 games, average rating 94.9
Big hits: 121 games, average rating 93.4
                                                 name  price  pct_pos_total
712                           Touhou Mystia's Izakaya   5.99             98
874                                       Baba Is You  14.99             98
764                 There Is No Game: Wrong Dimension   9.74             97
799                                              DUSK   6.79             97
963  Milk outside a bag of milk outside a bag of milk   8.99             97


---
## The recommendation

*(Example choices: `GENRE = "Indie"`, `MAX_PRICE = 20`, `SYSTEM = "mac"`.)*

**Sub-question 1 (Cell 2):** 154 Indie games with at least 85% positive reviews cost $20 or less and run on a Mac. The student has plenty of good choices, so the hard part is picking one.

**Sub-question 2 (Cell 5):** CheapShark checks the 5 best-reviewed paid games that fit. Each card shows today's price next to the regular price and the lowest price ever, and the "On sale today" list shows which are cheaper than usual right now.

**Sub-question 3 (Cell 6):** of the 148 games with a known owner count, 27 are hidden gems with under 1 million owners. They average 94.9% positive reviews, a bit higher than the 121 big hits (93.4%). The best ones include Touhou Mystia's Izakaya and Baba Is You, both 98% positive.

**For the student:** buy a game from the "On sale today" list, especially one whose price today is at or near its lowest price ever. If none is on sale this week, try a hidden gem instead: Touhou Mystia's Izakaya is 98% positive and costs only $5.99.